In [10]:
# Install ipywidgets if needed
import subprocess
subprocess.run(["pip", "install", "ipywidgets"])
print("Done")

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Done


In [11]:
# Import libraries
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from imblearn.combine import SMOTETomek

In [12]:
# Load and train the tuned hybrid model
# Using the best parameters found during hyperparameter tuning
print("Loading model... please wait")

df = pd.read_csv("/home/ae7ba225-76ef-4005-aee2-7847a70630ed/SilentTelecomChurn/engineered_features.csv")

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

smotetomek = SMOTETomek(random_state=42)
X_train_balanced, y_train_balanced = smotetomek.fit_resample(X_train, y_train)

# Best parameters from hyperparameter tuning
lr = LogisticRegression(C=0.01, solver="lbfgs", random_state=42)
svm = SVC(C=10, kernel="linear", probability=True, random_state=42)
xgb = XGBClassifier(learning_rate=0.1, max_depth=5, n_estimators=100,
                    random_state=42, eval_metric="logloss")

model = VotingClassifier(
    estimators=[("lr", lr), ("svm", svm), ("xgb", xgb)],
    voting="soft"
)

model.fit(X_train_balanced, y_train_balanced)
print("Model loaded and ready")

Loading model... please wait
Model loaded and ready


In [13]:
# Build the input widgets — best UI component for each feature

# Title
title = widgets.HTML(
    value="<h2 style='color:#2c3e50;'>📡 Silent Telco Churn Predictor</h2>"
          "<p style='color:#7f8c8d;'>Enter customer details below and click Predict</p>"
          "<hr>"
)

# Tenure Group — Dropdown (3 distinct loyalty categories)
tenure_dropdown = widgets.Dropdown(
    options=[
        ("New — 0 to 12 months", 0),
        ("Established — 13 to 36 months", 1),
        ("Loyal — More than 36 months", 2)
    ],
    value=0,
    description="Tenure Group:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="400px")
)

# Charge Level — Dropdown (3 distinct pricing categories)
charge_dropdown = widgets.Dropdown(
    options=[
        ("Low — Below GHS 35", 0),
        ("Medium — GHS 35 to 69", 1),
        ("High — GHS 70 and above", 2)
    ],
    value=0,
    description="Charge Level:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="400px")
)

# Multiple Lines — Radio buttons (binary Yes or No)
multiple_lines_radio = widgets.RadioButtons(
    options=[("No", 0), ("Yes", 1)],
    value=0,
    description="Multiple Lines:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="400px")
)

# Tech Support — Radio buttons (binary Yes or No)
tech_support_radio = widgets.RadioButtons(
    options=[("No", 0), ("Yes", 1)],
    value=0,
    description="Tech Support:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="400px")
)

# Internet Service — Dropdown (3 distinct service types)
internet_dropdown = widgets.Dropdown(
    options=[
        ("No Internet Service", 0),
        ("DSL / Fixed Home Broadband", 1),
        ("Fiber Optic Broadband", 2)
    ],
    value=0,
    description="Internet:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="400px")
)

# Online Security — Radio buttons (binary Yes or No)
online_security_radio = widgets.RadioButtons(
    options=[("No", 0), ("Yes", 1)],
    value=0,
    description="Online Security:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="400px")
)

# Contract — Dropdown (3 distinct contract types)
contract_dropdown = widgets.Dropdown(
    options=[
        ("Month-to-Month", 0),
        ("One Year", 1),
        ("Two Year", 2)
    ],
    value=0,
    description="Contract:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="400px")
)

# Predict button
predict_button = widgets.Button(
    description="Predict Churn Risk",
    button_style="primary",
    layout=widgets.Layout(width="300px", height="45px")
)

# Output area
output = widgets.Output()

In [14]:
# Define what happens when the predict button is clicked
def on_predict_clicked(b):
    with output:
        clear_output()

        # Collect all input values from updated widgets
        customer_input = np.array([[
            tenure_dropdown.value,
            charge_dropdown.value,
            multiple_lines_radio.value,
            tech_support_radio.value,
            internet_dropdown.value,
            online_security_radio.value,
            contract_dropdown.value
        ]])

        # Get prediction and probability
        prediction = model.predict(customer_input)[0]
        probability = model.predict_proba(customer_input)[0]

        churn_prob = round(probability[1] * 100, 1)
        retain_prob = round(probability[0] * 100, 1)

        print("=" * 55)

        if prediction == 1:
            print("⚠️  HIGH CHURN RISK")
            print(f"    This customer is likely to silently leave")
            print(f"    Churn Probability    : {churn_prob}%")
            print(f"    Retention Probability: {retain_prob}%")
            print()
            print("Risk Factors:")

            if contract_dropdown.value == 0:
                print("  • Month-to-Month contract — no long term commitment")
            if internet_dropdown.value == 2:
                print("  • Fiber Optic — highest churn risk internet tier")
            if tenure_dropdown.value == 0:
                print("  • New customer — loyalty not yet established")
            if online_security_radio.value == 0:
                print("  • No Online Security — fewer services tying them in")
            if tech_support_radio.value == 0:
                print("  • No Tech Support — less provider dependency")
            if charge_dropdown.value == 2:
                print("  • High charges — more likely to compare alternatives")

        else:
            print("✅  LOW CHURN RISK")
            print(f"    This customer is likely to stay with the network")
            print(f"    Retention Probability: {retain_prob}%")
            print(f"    Churn Probability    : {churn_prob}%")
            print()
            print("Retention Factors:")

            if contract_dropdown.value == 2:
                print("  • Two Year contract — strong commitment to provider")
            if contract_dropdown.value == 1:
                print("  • One Year contract — meaningful commitment to provider")
            if tenure_dropdown.value == 2:
                print("  • Loyal customer — over 36 months with the provider")
            if tenure_dropdown.value == 1:
                print("  • Established customer — 13 to 36 months with provider")
            if online_security_radio.value == 1:
                print("  • Has Online Security — additional switching cost")
            if tech_support_radio.value == 1:
                print("  • Has Tech Support — dependent on provider for support")
            if internet_dropdown.value == 0:
                print("  • No internet service — basic user with lower churn rate")

        print()
        print("=" * 55)
        print()
        print("SHAP Feature Importance (from analysis):")
        print("  1. Contract          — 1.2136")
        print("  2. Internet Service  — 0.7193")
        print("  3. Tenure Group      — 0.4916")
        print("  4. Charge Level      — 0.2042")
        print("  5. Online Security   — 0.1684")
        print("  6. Tech Support      — 0.1565")
        print("  7. Multiple Lines    — 0.1208")
        print("=" * 55)

# Connect button to function
predict_button.on_click(on_predict_clicked)

In [15]:
# Display the full UI
display(widgets.VBox([
    title,
    widgets.HTML("<b>Customer Loyalty</b>"),
    tenure_dropdown,
    widgets.HTML("<br><b>Pricing</b>"),
    charge_dropdown,
    widgets.HTML("<br><b>Services</b>"),
    multiple_lines_radio,
    tech_support_radio,
    internet_dropdown,
    online_security_radio,
    widgets.HTML("<br><b>Contract</b>"),
    contract_dropdown,
    widgets.HTML("<br>"),
    predict_button,
    widgets.HTML("<br>"),
    output
]))